## HumanEvals

We repeat the same thing here, but now instead of comparing the nll of the target and the predicted tokens, we take the probabilities of the predicted token and then sum their logs together. 

In [1]:
!pip install -qU torch  
!pip install -qU transformers
!pip install -qU datasets altair huggingface_hub tqdm accelerate python-dotenv
!pip install -qU sentencepiece bitsandbytes
!pip install -qU git+https://github.com/huggingface/accelerate.git
!pip install -qU protobuf

import torch as t
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from datasets import load_dataset
import altair as alt
from huggingface_hub import notebook_login
from tqdm import tqdm

import os
from dotenv import load_dotenv
from collections import defaultdict

device = "cuda" if t.cuda.is_available() else "cpu"

In [2]:
import sys
stdout = sys.stdout

In [3]:
!nvidia-smi

Wed Jun  5 15:17:19 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:00:06.0 Off |                    0 |
| N/A   29C    P0             52W /  300W |       3MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
HF_TOKEN = os.getenv("HF_TOKEN")
notebook_login()

In [5]:
GPT2_MODEL = "openai-community/gpt2-medium"
MISTRAL_MODEL = "mistralai/Mistral-7B-v0.3"
LLAMA3_MODEL = "meta-llama/Meta-Llama-3-8B"
LLAMA2_MODEL = "meta-llama/Llama-2-7b-hf"
GEMMA_MODEL = "google/gemma-7b"
OPENCHAT_MODEL = "openchat/openchat-3.6-8b-20240522"

## Dataset

We're using the original humaneval dataset (comment -> function). 

In [35]:
dataset = load_dataset("THUDM/humaneval-x", "cpp", split="test", trust_remote_code=True)
prompts = dataset["prompt"]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

## Calculating the Perplexity

The function to calculate the perplexity is slightly different because of the change in how I calculate the problabilities. 

In [69]:
from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper

def top_k_top_p_filtering(
    logits: t.FloatTensor,
    top_k: int = 0,
    top_p: float = 1.0,
    filter_value: float = -float("Inf"),
    min_tokens_to_keep: int = 1,
) -> t.FloatTensor:

    if top_k > 0:
        logits = TopKLogitsWarper(top_k=top_k, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    if 0 <= top_p <= 1.0:
        logits = TopPLogitsWarper(top_p=top_p, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    return logits

def calculate_mean_nll_loss(model, tokenizer, prompt):
    num_sampled = 0
    nll_vals = []
    # convert the input ids to a list of ints (this is a running list of input_ids)
    input_ids = tokenizer.encode(prompts[8], return_tensors='pt').tolist()[0]

    while True:
        input_ids_t = t.tensor([input_ids]).to(device)
        outputs = model(input_ids_t)
        # Get logits from last layer
        last_layer_logits = outputs.logits[:, -1, :]
    
        # Extract the 
        top_logits = top_k_top_p_filtering(last_layer_logits, top_k=1, top_p=1.0)
        probs = F.softmax(top_logits, dim=-1)
        token_idx = t.argmax(probs, dim=-1).item()
        token_prob = probs[0,token_idx]
        nll = -1 * t.log(token_prob).item()
        decoded_token = tokenizer.decode(t.tensor([token_idx]), skip_special_tokens=False)
        nll_vals.append(nll)
        input_ids.append(token_idx)
        num_sampled += 1
        
        # Once the model is done predicting, we calculate the perplexity by taking the 
        # exp of the mean nll values.
        if decoded_token == tokenizer.eos_token or token_idx == tokenizer.eos_token_id:
            ppl = t.exp(t.tensor(nll_vals).mean())
            return ppl, nll_vals


humanevals_ppls = defaultdict(list)

**Llama3 Evals**

In [74]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=t.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(LLAMA3_MODEL,
                                             quantization_config=bnb_config,
                                             device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(LLAMA3_MODEL,
                                         add_eos_token=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [9]:
!nvidia-smi

Wed Jun  5 15:17:51 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:00:06.0 Off |                    0 |
| N/A   30C    P0             71W /  300W |    6027MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
for i in tqdm(range(len(prompts))):
    ppl, _ = calculate_mean_nll_loss(model, tokenizer, prompts[i])
    humanevals_ppls[LLAMA3_MODEL].append(ppl)

 55%|█████▍    | 90/164 [17:52<14:44, 11.95s/it]

In [ ]:
humanevals_ppls

**Llama2 Evals**

In [61]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=t.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(LLAMA2_MODEL,
                                             quantization_config=bnb_config,
                                             device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(LLAMA2_MODEL,
                                         add_eos_token=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
num_sampled = 0
nll_vals = []
# convert the input ids to a list of ints (this is a running list of input_ids)
input_ids = tokenizer.encode(prompts[12], return_tensors='pt').tolist()[0]

while True:
    input_ids_t = t.tensor([input_ids]).to(device)
    outputs = model(input_ids_t)
    # Get logits from last layer
    last_layer_logits = outputs.logits[:, -1, :]

    # Extract the 
    top_logits = top_k_top_p_filtering(last_layer_logits, top_k=10, top_p=1.0)
    probs = F.softmax(top_logits, dim=-1)
    token_idx = t.argmax(probs, dim=-1).item()
    token_prob = probs[0,token_idx]
    nll = -1 * t.log(token_prob).item()
    decoded_token = tokenizer.decode(t.tensor([token_idx]), skip_special_tokens=False)
    print(f"Predicted {token_idx} with probability {token_prob} -- {token_idx} => {decoded_token}")
    nll_vals.append(nll)
    input_ids.append(token_idx)
    num_sampled += 1
    
    # Once the model is done predicting, we calculate the perplexity by taking the 
    # exp of the mean nll values.
    if decoded_token == tokenizer.eos_token or token_idx == tokenizer.eos_token_id:
        ppl = t.exp(t.tensor(nll_vals).mean())
        break 

print(ppl)

In [ ]:
for i in tqdm(range(len(prompts))):
    ppl, _ = calculate_mean_nll_loss(model, tokenizer, prompts[i])
    humanevals_ppls[LLAMA2_MODEL].append(ppl)

**Mistral Evals**

In [63]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=t.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL,
                                             quantization_config=bnb_config,
                                             device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL,
                                         add_eos_token=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [66]:
num_sampled = 0
nll_vals = []
# convert the input ids to a list of ints (this is a running list of input_ids)
input_ids = tokenizer.encode(prompts[5], return_tensors='pt').tolist()[0]

while True:
    input_ids_t = t.tensor([input_ids]).to(device)
    outputs = model(input_ids_t)
    # Get logits from last layer
    last_layer_logits = outputs.logits[:, -1, :]

    # Extract the 
    top_logits = top_k_top_p_filtering(last_layer_logits, top_k=1, top_p=1.0)
    probs = F.softmax(top_logits, dim=-1)
    token_idx = t.argmax(probs, dim=-1).item()
    token_prob = probs[0,token_idx]
    nll = -1 * t.log(token_prob).item()
    decoded_token = tokenizer.decode(t.tensor([token_idx]), skip_special_tokens=False)
    print(f"Predicted {token_idx} with probability {token_prob} -- {token_idx} => {decoded_token}")
    nll_vals.append(nll)
    input_ids.append(token_idx)
    num_sampled += 1
    
    # Once the model is done predicting, we calculate the perplexity by taking the 
    # exp of the mean nll values.
    if decoded_token == tokenizer.eos_token or token_idx == tokenizer.eos_token_id:
        ppl = t.exp(t.tensor(nll_vals).mean())
        break 

print(ppl)

Predicted 2460 with probability 1.0 -- 2460 => ве
Predicted 14902 with probability 1.0 -- 14902 => ктор
Predicted 29557 with probability 1.0 -- 29557 => <
Predicted 1269 with probability 1.0 -- 1269 => int
Predicted 29535 with probability 1.0 -- 29535 => >
Predicted 1972 with probability 1.0 -- 1972 => result
Predicted 29513 with probability 1.0 -- 29513 => ;
Predicted 781 with probability 1.0 -- 781 => 

Predicted 3055 with probability 1.0 -- 3055 =>   
Predicted 1122 with probability 1.0 -- 1122 => for
Predicted 29500 with probability 1.0 -- 29500 => (
Predicted 1269 with probability 1.0 -- 1269 => int
Predicted 1381 with probability 1.0 -- 1381 => i
Predicted 1095 with probability 1.0 -- 1095 => =
Predicted 29473 with probability 1.0 -- 29473 => 
Predicted 29502 with probability 1.0 -- 29502 => 0
Predicted 29513 with probability 1.0 -- 29513 => ;
Predicted 1381 with probability 1.0 -- 1381 => i
Predicted 1291 with probability 1.0 -- 1291 => <
Predicted 6319 with probability 1.0 -- 6

KeyboardInterrupt: 

In [70]:
for i in tqdm(range(len(prompts))):
    ppl, _ = calculate_mean_nll_loss(model, tokenizer, prompts[i])
    humanevals_ppls[MISTRAL_MODEL].append(ppl)

100%|██████████| 164/164 [48:52<00:00, 17.88s/it]


In [72]:
humanevals_ppls

defaultdict(list,
            {'mistralai/Mistral-7B-v0.3': [tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.0041),
              tensor(1.